# Paper — Data Construction Statistics

All counts used in the *Database Construction* paragraph of the paper. Queries run with **Polars** (via `read_database`) against `data/humans_clean.duckdb`.

Used to fill the `(QX)` / `X` placeholders and to produce **Table X. Data extracted at the beginning of the pipeline**.

In [ ]:
import os
import duckdb
from pathlib import Path

import polars as pl

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DB_PATH = ROOT / "data" / "humans_clean.duckdb"
assert DB_PATH.exists(), DB_PATH
os.makedirs(ROOT / 'paper', exist_ok=True)

con = duckdb.connect(DB_PATH, read_only=True)

def scalar(sql: str):
    return con.execute(sql).pl().item()

def df(sql: str) -> pl.DataFrame:
    return con.execute(sql).pl()

## 1. Core counts

In [2]:
core = {
    "individuals":              scalar("SELECT COUNT(*) FROM individuals"),
    "places (all)":             scalar("SELECT COUNT(*) FROM places"),
    "places (urban settlement)":scalar("SELECT COUNT(*) FROM places WHERE is_urban_settlement = 1"),
    "occupations":              scalar("SELECT COUNT(*) FROM occupations"),
    "writing_languages":        scalar("SELECT COUNT(*) FROM writing_languages"),
    "country_of_citizenship":   scalar("SELECT COUNT(*) FROM country_of_citizenship"),
    "identifiers (records)":    scalar("SELECT COUNT(*) FROM identifiers"),
    "identifier_types":         scalar("SELECT COUNT(*) FROM identifier_types"),
    "wikimedia_links":          scalar("SELECT COUNT(*) FROM wikimedia_links"),
    "works":                    scalar("SELECT COUNT(*) FROM works"),
    "individuals_cliopatria":   scalar("SELECT COUNT(*) FROM individuals_cliopatria"),
    "polities_cliopatria":      scalar("SELECT COUNT(*) FROM polities_cliopatria"),
}
core_df = pl.DataFrame({"entity": list(core.keys()), "count": list(core.values())})
core_df

entity,count
str,i64
"""individuals""",13002897
"""places (all)""",314675
"""places (urban settlement)""",257269
"""occupations""",18227
"""writing_languages""",524
…,…
"""identifier_types""",5152
"""wikimedia_links""",15544183
"""works""",38554301


## 2. Coverage of each individual

In [3]:
coverage = {
    "any place (birth/death/citizenship)": scalar(
        "SELECT COUNT(*) FROM individuals WHERE "
        "(birthcity_en IS NOT NULL AND birthcity_en != '') OR "
        "(deathcity_en IS NOT NULL AND deathcity_en != '') OR "
        "(country_of_citizenship_en IS NOT NULL AND country_of_citizenship_en != '')"
    ),
    "any date (birth/death/floruit)": scalar(
        "SELECT COUNT(*) FROM individuals WHERE "
        "(birthdate IS NOT NULL AND birthdate != '') OR "
        "(deathdate IS NOT NULL AND deathdate != '') OR "
        "(floruit_date IS NOT NULL AND floruit_date != '')"
    ),
    "any catalog identifier":        scalar("SELECT COUNT(*) FROM individuals WHERE identifiers_count >= 1"),
    "any Wikimedia sitelink":        scalar("SELECT COUNT(*) FROM individuals WHERE wikimedia_links_count >= 1"),
    "any identifier OR sitelink":    scalar("SELECT COUNT(*) FROM individuals WHERE identifiers_count >= 1 OR wikimedia_links_count >= 1"),
    "place AND date AND (id OR sitelink)": scalar(
        "SELECT COUNT(*) FROM individuals WHERE "
        "((birthcity_en IS NOT NULL AND birthcity_en != '') OR (deathcity_en IS NOT NULL AND deathcity_en != '') OR (country_of_citizenship_en IS NOT NULL AND country_of_citizenship_en != '')) AND "
        "((birthdate IS NOT NULL AND birthdate != '') OR (deathdate IS NOT NULL AND deathdate != '') OR (floruit_date IS NOT NULL AND floruit_date != '')) AND "
        "(identifiers_count >= 1 OR wikimedia_links_count >= 1)"
    ),
}
coverage_df = pl.DataFrame({"criterion": list(coverage.keys()), "individuals": list(coverage.values())})
coverage_df

criterion,individuals
str,i64
"""any place (birth/death/citizen…",6376666
"""any date (birth/death/floruit)""",7812877
"""any catalog identifier""",11503576
"""any Wikimedia sitelink""",4959581
"""any identifier OR sitelink""",12059591
"""place AND date AND (id OR site…",5107906


## 3. Removed entries (non-historical filtering)

In [4]:
non_human_total = scalar("SELECT COUNT(*) FROM individuals WHERE non_human = 1")
fictional_polity_qids = df(
    "SELECT wikidata_id, name_en, count, instance_labels FROM country_of_citizenship "
    "WHERE LOWER(instance_labels) LIKE '%fictional%' OR LOWER(instance_labels) LIKE '%mytholog%'"
)
removed = pl.DataFrame({
    "filter": [
        "P31 = fictional/mythical/legendary/deity (non_human flag)",
        "country_of_citizenship instance = fictional polity (Q-IDs touched)",
    ],
    "count": [non_human_total, len(fictional_polity_qids)],
})
removed

filter,count
str,i64
"""P31 = fictional/mythical/legen…",367
"""country_of_citizenship instanc…",350


## 4. Top instance-of types of works

In [5]:
df(
    "SELECT instance_of_en AS work_type, COUNT(*) AS n "
    "FROM works WHERE instance_of_en IS NOT NULL "
    "GROUP BY instance_of_en ORDER BY n DESC LIMIT 20"
)

work_type,n
str,i64
"""scholarly article""",32173225
"""painting""",740284
"""film""",630308
"""version, edition or translatio…",582189
"""literary work""",433964
…,…
"""scholarly article|case report""",116445
"""print""",115950
"""written work""",113259


## 5. Table X. Data extracted at the beginning of the pipeline

Single summary table for the paper.

In [6]:
TOTAL = scalar("SELECT COUNT(*) FROM individuals")

# Per-individual coverage of each Wikidata property (at least one value)
ind_with = {
    "birthplace":  scalar("SELECT COUNT(*) FROM individuals WHERE birthcity_en IS NOT NULL AND birthcity_en != ''"),
    "deathplace":  scalar("SELECT COUNT(*) FROM individuals WHERE deathcity_en IS NOT NULL AND deathcity_en != ''"),
    "citizenship": scalar("SELECT COUNT(*) FROM individuals WHERE country_of_citizenship_en IS NOT NULL AND country_of_citizenship_en != ''"),
    "occupation":  scalar("SELECT COUNT(*) FROM individuals WHERE occupations_en IS NOT NULL AND occupations_en != ''"),
    "writing_lang":scalar("SELECT COUNT(*) FROM individuals WHERE writing_language_name_en IS NOT NULL AND writing_language_name_en != ''"),
    "birthdate":   scalar("SELECT COUNT(*) FROM individuals WHERE birthdate IS NOT NULL AND birthdate != ''"),
    "deathdate":   scalar("SELECT COUNT(*) FROM individuals WHERE deathdate IS NOT NULL AND deathdate != ''"),
    "floruit":     scalar("SELECT COUNT(*) FROM individuals WHERE floruit_date IS NOT NULL AND floruit_date != ''"),
    "gender":      scalar("SELECT COUNT(*) FROM individuals WHERE gender IS NOT NULL AND gender != ''"),
    "identifier":  scalar("SELECT COUNT(*) FROM individuals WHERE identifiers_count >= 1"),
    "sitelink":    scalar("SELECT COUNT(*) FROM individuals WHERE wikimedia_links_count >= 1"),
    "work":        scalar("SELECT COUNT(*) FROM individuals WHERE number_of_works >= 1"),
    "polity":      scalar("SELECT COUNT(DISTINCT wikidata_id) FROM individuals_cliopatria"),
}

# Cardinality of distinct values referenced by each property
uniq = {
    "birthplace":  scalar("SELECT COUNT(DISTINCT birthcity_id) FROM individuals_keys WHERE birthcity_id IS NOT NULL AND birthcity_id != ''"),
    "deathplace":  scalar("SELECT COUNT(DISTINCT deathcity_id) FROM individuals_keys WHERE deathcity_id IS NOT NULL AND deathcity_id != ''"),
    "gender":      scalar("SELECT COUNT(DISTINCT gender) FROM individuals WHERE gender IS NOT NULL AND gender != ''"),
    "work_type":   scalar("SELECT COUNT(DISTINCT instance_of_en) FROM works WHERE instance_of_en IS NOT NULL"),
    "wm_sites":    scalar("SELECT COUNT(DISTINCT site) FROM wikimedia_links"),
    "polities":    scalar("SELECT COUNT(*) FROM polities_cliopatria"),
}

# rows: (entity, source table, property, count, individuals_with_at_least_one_value, unique_values)
rows = [
    ("Individuals (humans, P31 = Q5)",                       "individuals",            "P31",     TOTAL,                                                          TOTAL,                                            TOTAL),
    ("Removed: fictional / mythical (P31)",                  "individuals (filtered)", "P31",     non_human_total,                                                None,                                             None),
    ("Place of birth",                                        "places / individuals",   "P19",     scalar("SELECT COUNT(*) FROM places"),                          ind_with["birthplace"],                           uniq["birthplace"]),
    ("Place of death",                                        "places / individuals",   "P20",     None,                                                           ind_with["deathplace"],                           uniq["deathplace"]),
    ("Places: urban settlements (cities)",                    "places",                 "P31",     scalar("SELECT COUNT(*) FROM places WHERE is_urban_settlement = 1"), None,                                        scalar("SELECT COUNT(*) FROM places WHERE is_urban_settlement = 1")),
    ("Country of citizenship",                                "country_of_citizenship", "P27",     scalar("SELECT COUNT(*) FROM country_of_citizenship"),          ind_with["citizenship"],                          scalar("SELECT COUNT(*) FROM country_of_citizenship")),
    ("Occupation",                                            "occupations",            "P106",    scalar("SELECT COUNT(*) FROM occupations"),                     ind_with["occupation"],                           scalar("SELECT COUNT(*) FROM occupations")),
    ("Writing language",                                      "writing_languages",      "P6886",   scalar("SELECT COUNT(*) FROM writing_languages"),               ind_with["writing_lang"],                         scalar("SELECT COUNT(*) FROM writing_languages")),
    ("Birth date",                                            "individuals",            "P569",    ind_with["birthdate"],                                          ind_with["birthdate"],                            None),
    ("Death date",                                            "individuals",            "P570",    ind_with["deathdate"],                                          ind_with["deathdate"],                            None),
    ("Floruit date",                                          "individuals",            "P1317",   ind_with["floruit"],                                            ind_with["floruit"],                              None),
    ("Gender",                                                "individuals",            "P21",     ind_with["gender"],                                             ind_with["gender"],                               uniq["gender"]),
    ("External identifier systems (catalogs)",                "identifier_types",       "—",       scalar("SELECT COUNT(*) FROM identifier_types"),                None,                                             scalar("SELECT COUNT(*) FROM identifier_types")),
    ("External identifier records",                           "identifiers",            "—",       scalar("SELECT COUNT(*) FROM identifiers"),                     ind_with["identifier"],                           scalar("SELECT COUNT(*) FROM identifier_types")),
    ("Wikimedia sitelinks",                                   "wikimedia_links",        "—",       scalar("SELECT COUNT(*) FROM wikimedia_links"),                 ind_with["sitelink"],                             uniq["wm_sites"]),
    ("Works (creative outputs)",                              "works",                  "—",       scalar("SELECT COUNT(*) FROM works"),                           ind_with["work"],                                 uniq["work_type"]),
    ("Cliopatria polities",                                   "polities_cliopatria",    "—",       uniq["polities"],                                               None,                                             uniq["polities"]),
    ("Individual–polity links (Cliopatria)",                  "individuals_cliopatria", "—",       scalar("SELECT COUNT(*) FROM individuals_cliopatria"),          ind_with["polity"],                               uniq["polities"]),
    ("Individuals with any place (birth/death/citizenship)",  "individuals",            "—",       coverage["any place (birth/death/citizenship)"],                coverage["any place (birth/death/citizenship)"],  None),
    ("Individuals with any date (birth/death/floruit)",       "individuals",            "—",       coverage["any date (birth/death/floruit)"],                     coverage["any date (birth/death/floruit)"],       None),
    ("Individuals with any catalog id or sitelink",           "individuals",            "—",       coverage["any identifier OR sitelink"],                         coverage["any identifier OR sitelink"],           None),
    ("Individuals with place + date + (id or sitelink)",      "individuals",            "—",       coverage["place AND date AND (id OR sitelink)"],                coverage["place AND date AND (id OR sitelink)"],  None),
]

def pct(c):
    return None if c is None else round(100.0 * c / TOTAL, 2)

table_x = pl.DataFrame(
    {
        "entity":           [r[0] for r in rows],
        "table":            [r[1] for r in rows],
        "property":         [r[2] for r in rows],
        "count":            [r[3] for r in rows],
        "individuals_with": [r[4] for r in rows],
        "pct_individuals":  [pct(r[4]) for r in rows],
        "unique_values":    [r[5] for r in rows],
    }
)
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=240):
    print(table_x)
table_x.write_csv(ROOT / "paper" / "table_x_data_extracted.csv")
table_x

shape: (22, 7)
┌─────────────────────────────────┬────────────────────────┬──────────┬──────────┬──────────────────┬─────────────────┬───────────────┐
│ entity                          ┆ table                  ┆ property ┆ count    ┆ individuals_with ┆ pct_individuals ┆ unique_values │
│ ---                             ┆ ---                    ┆ ---      ┆ ---      ┆ ---              ┆ ---             ┆ ---           │
│ str                             ┆ str                    ┆ str      ┆ i64      ┆ i64              ┆ f64             ┆ i64           │
╞═════════════════════════════════╪════════════════════════╪══════════╪══════════╪══════════════════╪═════════════════╪═══════════════╡
│ Individuals (humans, P31 = Q5)  ┆ individuals            ┆ P31      ┆ 13002897 ┆ 13002897         ┆ 100.0           ┆ 13002897      │
│ Removed: fictional / mythical … ┆ individuals (filtered) ┆ P31      ┆ 367      ┆ null             ┆ null            ┆ null          │
│ Place of birth                 

entity,table,property,count,individuals_with,pct_individuals,unique_values
str,str,str,i64,i64,f64,i64
"""Individuals (humans, P31 = Q5)""","""individuals""","""P31""",13002897,13002897,100.0,13002897
"""Removed: fictional / mythical …","""individuals (filtered)""","""P31""",367,null,null,null
"""Place of birth""","""places / individuals""","""P19""",314675,3724787,28.65,275526
"""Place of death""","""places / individuals""","""P20""",null,1585483,12.19,130614
"""Places: urban settlements (cit…","""places""","""P31""",257269,null,null,257269
…,…,…,…,…,…,…
"""Individual–polity links (Cliop…","""individuals_cliopatria""","""—""",6128228,6128228,47.13,1604
"""Individuals with any place (bi…","""individuals""","""—""",6376666,6376666,49.04,null
"""Individuals with any date (bir…","""individuals""","""—""",7812877,7812877,60.09,null


## 6. Markdown render of Table X (paste-ready)

In [7]:
def fmt(n):
    return "" if n is None else f"{n:,}"

def fmt_pct(p):
    return "" if p is None else f"{p:.2f}%"

lines = [
    "| Entity | Table | Wikidata property | Count | Individuals with ≥1 value | % of individuals | Unique values |",
    "|---|---|---|---:|---:|---:|---:|",
]
for r in table_x.iter_rows(named=True):
    lines.append(
        f"| {r['entity']} | `{r['table']}` | {r['property']} | "
        f"{fmt(r['count'])} | {fmt(r['individuals_with'])} | {fmt_pct(r['pct_individuals'])} | "
        f"{fmt(r['unique_values'])} |"
    )
md = "\n".join(lines)
print(md)
(ROOT / "paper" / "table_x_data_extracted.md").write_text(md + "\n")

| Entity | Table | Wikidata property | Count | Individuals with ≥1 value | % of individuals | Unique values |
|---|---|---|---:|---:|---:|---:|
| Individuals (humans, P31 = Q5) | `individuals` | P31 | 13,002,897 | 13,002,897 | 100.00% | 13,002,897 |
| Removed: fictional / mythical (P31) | `individuals (filtered)` | P31 | 367 |  |  |  |
| Place of birth | `places / individuals` | P19 | 314,675 | 3,724,787 | 28.65% | 275,526 |
| Place of death | `places / individuals` | P20 |  | 1,585,483 | 12.19% | 130,614 |
| Places: urban settlements (cities) | `places` | P31 | 257,269 |  |  | 257,269 |
| Country of citizenship | `country_of_citizenship` | P27 | 4,571 | 5,570,031 | 42.84% | 4,571 |
| Occupation | `occupations` | P106 | 18,227 | 9,032,161 | 69.46% | 18,227 |
| Writing language | `writing_languages` | P6886 | 524 | 225,825 | 1.74% | 524 |
| Birth date | `individuals` | P569 | 7,511,293 | 7,511,293 | 57.77% |  |
| Death date | `individuals` | P570 | 3,573,392 | 3,573,392 | 27.48% |  |
| 

2121